## Pydantic Output Parser

pydantic output parser is a structured output parser in langchain that uses pydantic models to enforce schema validation 
when processing LLM responses.

## Why use pydantic output parser ?

1. Strict Schema enforcement - ensures thst LLM responses follow a well-defined structure.
2. Type Safety - Automatically converts LLM outputs into Python objects.
3. Easy Validation - Uses Pydantic's built-in validation to catch incorrect or missing data.
4. Seamless Integration - Works well with other Langchain components.

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser          
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel , Field

import langchain



load_dotenv()

model = ChatGoogleGenerativeAI (
    model="gemini-2.5-flash",
    temperature=0,
)

c:\Users\KIIT0001\Desktop\Langchain-from-Scratch\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [8]:
class Person(BaseModel) :
    name : str = Field(description='Name of the person')
    age : int = Field(gt=18 , description='Age of the person')
    city : str =  Field(description='Name of the city the person belongs to')

In [9]:
parser = PydanticOutputParser(pydantic_object=Person)

In [12]:
template = PromptTemplate(
    template='Generate the name , age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

In [13]:
prompt = template.invoke({'place' : 'indian'})

In [18]:
print(prompt.text)

Generate the name , age and city of a fictional indian person 
 The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "Name of the person", "title": "Name", "type": "string"}, "age": {"description": "Age of the person", "exclusiveMinimum": 18, "title": "Age", "type": "integer"}, "city": {"description": "Name of the city the person belongs to", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}
```


In [14]:
result  = model.invoke(prompt)

In [15]:
final_result = parser.parse(result.content)

In [16]:
print(final_result)

name='Priya Sharma' age=28 city='Bengaluru'


## using chains

In [19]:
chain = template | model | parser

In [20]:
result = chain.invoke({'place' : 'sri lankan'})

In [21]:
print(result)

name='Nimal Fernando' age=32 city='Kandy'
